# IQB-Edu: From Policy Goals to Connectivity Standards
### Moldova School Connectivity — January 2026

> **How to use this notebook**
> Press **Run → Run All Cells** (or Shift+Enter through each cell).
> The interactive panel will appear at the bottom of the page.
> No coding needed — everything is controlled by dropdowns and sliders.

---

## What question does this tool answer?

> *"Given what we want schools to be able to do online, how many schools in Moldova can actually do it?"*

Internet connectivity is not equally valuable for every educational goal.
A school that can comfortably support web browsing and audio may still struggle with
live video conferencing. A connection that looks adequate on a typical day may fail
students during peak hours.

The **Internet Quality Barometer for Education (IQB-Edu)** translates raw network
measurements — download speed, upload speed, latency, and packet loss — into a
single score (0–1) that reflects how well a school's connection supports a given
set of educational activities.

This tool puts the policy choices in your hands:

| Control | What it does |
|---|---|
| **Policy Scenario** | Pre-set bundles that reflect common educational goals |
| **Use Case Priorities** | How much each online activity matters for your context |
| **Performance Standard** | Whether to evaluate typical days, worst-case days, or peak capacity |
| **IQB-Edu Scores for Schools** | Pick a threshold to see how many schools perform above it |

The charts update the moment you change any setting, showing you exactly how many
schools clear the bar — and which ones don't.

---


In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings, copy
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from iqbedu.data import IQBEduData
from iqb.calculator import IQBCalculator, IQB_CONFIG


In [2]:
# ── Data loading ────────────────────────────────────────────────────────────
STATS_DATA_PATH = "./data/IQB_Edu_MD_2026-01-01_2026-02-01_stats.csv"
GEO_DATA_PATH   = "./data/giga_MD_schools_metadata.csv"

df_stats = pd.read_csv(STATS_DATA_PATH)
df_geo   = pd.read_csv(GEO_DATA_PATH)
data     = IQBEduData(df=df_stats)
IQB_USE_CASES = list(IQB_CONFIG["use cases"].keys())

print(f"✓  {len(data.schools)} schools loaded  |  "
      f"{int(df_stats['download_sample_count'].sum()):,} M-Lab measurements  |  "
      f"Period: January 2026")


✓  57 schools loaded  |  3,509 M-Lab measurements  |  Period: January 2026


In [3]:
# ── Policy scenario definitions ───────────────────────────────────────────────
POLICY_SCENARIOS = {
    "Basic Digital Learning": {
        "description": (
            "Covers the essentials: web research, reading, and listening. "
            "Appropriate for schools in bandwidth-constrained environments "
            "where the priority is foundational digital access for all students."
        ),
        "weights": {
            "web browsing": 1.0, "audio streaming": 0.8, "video streaming": 0.2,
            "video conferencing": 0.1, "online backup": 0.3, "gaming": 0.0,
        },
        "percentile": 50,
    },
    "Hybrid Classroom": {
        "description": (
            "Live video lessons and collaborative tools are central. "
            "Suitable for schools expected to deliver synchronous remote "
            "or blended instruction alongside in-person teaching."
        ),
        "weights": {
            "web browsing": 0.8, "audio streaming": 0.5, "video streaming": 1.0,
            "video conferencing": 1.0, "online backup": 0.3, "gaming": 0.0,
        },
        "percentile": 50,
    },
    "Equity / Low-Bandwidth": {
        "description": (
            "Evaluated under worst-quarter conditions — the hardest test. "
            "Ensures that even poorly-served schools can reliably deliver "
            "lightweight digital content to every student every day."
        ),
        "weights": {
            "web browsing": 1.0, "audio streaming": 1.0, "video streaming": 0.0,
            "video conferencing": 0.0, "online backup": 0.5, "gaming": 0.0,
        },
        "percentile": 25,
    },
    "Universal Digital Education": {
        "description": (
            "Equal weight across all meaningful educational use cases. "
            "A balanced national baseline — no single application is "
            "prioritised over others."
        ),
        "weights": {
            "web browsing": 1.0, "audio streaming": 1.0, "video streaming": 1.0,
            "video conferencing": 1.0, "online backup": 1.0, "gaming": 0.0,
        },
        "percentile": 50,
    },
    "Future-Ready": {
        "description": (
            "Ambitious standards across all use cases, including gaming "
            "and near-peak capacity. A forward-looking benchmark for "
            "infrastructure investment planning."
        ),
        "weights": {
            "web browsing": 1.0, "audio streaming": 1.0, "video streaming": 1.0,
            "video conferencing": 1.0, "online backup": 0.8, "gaming": 0.5,
        },
        "percentile": 95,
    },
    "Custom": {
        "description": "Use the sliders below to define your own policy priorities.",
        "weights": {
            "web browsing": 0.8, "audio streaming": 0.5, "video streaming": 0.8,
            "video conferencing": 0.8, "online backup": 0.5, "gaming": 0.0,
        },
        "percentile": 50,
    },
}

UC_LABELS = {
    "web browsing":       "🌐  Web Browsing",
    "audio streaming":    "🎵  Audio Streaming",
    "video streaming":    "📺  Video Streaming",
    "video conferencing": "📹  Video Conferencing",
    "online backup":      "☁️   Cloud Storage / Backup",
    "gaming":             "🎮  Educational Gaming",
}

PERC_OPTIONS = {
    "Connection capacity  (near-best — p95)":                   95,
    "Typical / average experience  (median — p50)":             50,
    "Conservative / difficult day experience  (p25)":           25,
}

# ── Score computation with caching ────────────────────────────────────────────
_score_cache = {}

def compute_scores(weights, percentile):
    key = (tuple(sorted(weights.items())), percentile)
    if key in _score_cache:
        return _score_cache[key]

    cfg  = copy.deepcopy(IQB_CONFIG)
    for uc in IQB_USE_CASES:
        cfg["use cases"][uc]["w"] = weights.get(uc, 0)
    calc = IQBCalculator(config=cfg)

    rows = []
    for sid in data.schools:
        mlab = data.to_iqb_data(sid, percentile=percentile)
        rows.append({"school_id": sid,
                     "score": calc.calculate_iqb_score({"m-lab": mlab.to_dict()})})

    df = pd.DataFrame(rows)
    df = pd.merge(df, data.df[["school_id", "giga_id_school",
                                "download_sample_count"]], on="school_id", how="left")
    df = pd.merge(df, df_geo, on="giga_id_school", how="left")
    _score_cache[key] = df
    return df

_uc_score_cache = {}

def compute_uc_scores(uc, percentile):
    key = (uc, percentile)
    if key in _uc_score_cache:
        return _uc_score_cache[key]
    cfg = copy.deepcopy(IQB_CONFIG)
    for u in IQB_USE_CASES:
        cfg["use cases"][u]["w"] = 1 if u == uc else 0
    calc = IQBCalculator(config=cfg)
    scores = [
        calc.calculate_iqb_score(
            {"m-lab": data.to_iqb_data(sid, percentile=percentile).to_dict()}
        )
        for sid in data.schools
    ]
    _uc_score_cache[key] = scores
    return scores

print("✓  Scenarios and computation helpers ready")


✓  Scenarios and computation helpers ready


---
## 🎛️  Policy Explorer

Configure the scenario below, then observe how school readiness changes.
The three charts and summary table update automatically.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
#  Run this cell to launch the interactive explorer.
# ═══════════════════════════════════════════════════════════════════════════
style  = {"description_width": "240px"}
layout = widgets.Layout(width="580px")

# ── Widgets ──────────────────────────────────────────────────────────────
scenario_dd = widgets.Dropdown(
    options=list(POLICY_SCENARIOS.keys()),
    value="Hybrid Classroom",
    description="Policy Scenario:",
    style=style, layout=layout,
)
scenario_desc = widgets.HTML(
    value=f"<p style='color:#444;font-style:italic;margin:2px 0 8px 244px'>"
          f"{POLICY_SCENARIOS['Hybrid Classroom']['description']}</p>"
)

weight_sliders = {
    uc: widgets.FloatSlider(
        value=POLICY_SCENARIOS["Hybrid Classroom"]["weights"][uc],
        min=0.0, max=1.0, step=0.1,
        description=UC_LABELS[uc],
        style=style, layout=layout,
        readout_format=".1f",
    )
    for uc in IQB_USE_CASES
}

perc_toggle = widgets.RadioButtons(
    options=list(PERC_OPTIONS.keys()),
    value="Typical / average experience  (median — p50)",
    description="Evaluate schools on:",
    style=style,
)

threshold_sl = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description="Threshold:",
    style=style, layout=layout,
    readout_format=".2f",
)

output = widgets.Output()

# ── Score-to-words helper ─────────────────────────────────────────────────
def score_label(s):
    if s >= 0.85: return "Ready"
    if s >= 0.65: return "Mostly ready"
    if s >= 0.45: return "Partially ready"
    return "Not ready"

def score_color(s, threshold):
    if s >= threshold:   return "#2ca02c"
    if s >= threshold * 0.75: return "#ff7f0e"
    return "#d62728"

# ── Scenario loader ───────────────────────────────────────────────────────
_loading = False

def load_scenario(change):
    global _loading
    _loading = True
    sc = POLICY_SCENARIOS[scenario_dd.value]
    scenario_desc.value = (
        f"<p style='color:#444;font-style:italic;margin:2px 0 8px 244px'>"
        f"{sc['description']}</p>"
    )
    for uc, sl in weight_sliders.items():
        sl.value = sc["weights"][uc]
    pk = next(k for k, v in PERC_OPTIONS.items() if v == sc["percentile"])
    perc_toggle.value = pk
    _loading = False
    refresh(None)

# ── Main refresh ──────────────────────────────────────────────────────────
def refresh(change):
    if _loading:
        return

    weights    = {uc: sl.value for uc, sl in weight_sliders.items()}
    percentile = PERC_OPTIONS[perc_toggle.value]
    threshold  = threshold_sl.value
    perc_short = perc_toggle.value.split("(")[1].rstrip(")")

    df       = compute_scores(weights, percentile)
    pct_ok   = (df["score"] >= threshold).mean() * 100
    pct_fail = 100 - pct_ok
    median_s = df["score"].median()

    df_sorted = df.sort_values("score").reset_index(drop=True)

    # ── Build figure ─────────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=3,
        column_widths=[0.27, 0.38, 0.35],
        subplot_titles=[
            "Priority Profile",
            "Score Distribution — all schools",
            "Schools Ranked by Score",
        ],
        specs=[[{"type": "polar"}, {"type": "xy"}, {"type": "xy"}]],
        horizontal_spacing=0.09,
    )

    # Radar — policy shape
    ucs   = IQB_USE_CASES
    r_v   = [weights[u] for u in ucs] + [weights[ucs[0]]]
    theta = [UC_LABELS[u] for u in ucs] + [UC_LABELS[ucs[0]]]
    fig.add_trace(go.Scatterpolar(
        r=r_v, theta=theta, fill="toself",
        fillcolor="rgba(33,102,172,0.20)",
        line=dict(color="rgb(33,102,172)", width=2),
        name="Priority weights",
    ), row=1, col=1)
    fig.update_polars(
        radialaxis=dict(range=[0, 1], showticklabels=False, ticks=""),
        angularaxis=dict(tickfont=dict(size=10)),
    )

    # Histogram
    fig.add_trace(go.Histogram(
        x=df["score"], nbinsx=16,
        marker_color="rgba(33,102,172,0.55)",
        name="Schools",
    ), row=1, col=2)
    # add_vline/add_hline with row/col trips on the Scatterpolar trace;
    # use add_shape with explicit axis references instead.
    # col 2 (histogram) → xref="x", yref="y domain"
    # col 3 (bar)       → xref="x2 domain", yref="y2"
    fig.add_shape(type="line", x0=threshold, x1=threshold, y0=0, y1=1,
                  xref="x", yref="y domain",
                  line=dict(dash="dash", color="crimson", width=1.5))
    fig.add_annotation(x=threshold, y=0.97, xref="x", yref="y domain",
                       text=f"min {threshold:.2f}", font=dict(color="crimson", size=10),
                       showarrow=False, yanchor="top", xanchor="left")
    fig.add_shape(type="line", x0=median_s, x1=median_s, y0=0, y1=1,
                  xref="x", yref="y domain",
                  line=dict(dash="dot", color="steelblue", width=1.5))
    fig.add_annotation(x=median_s, y=0.82, xref="x", yref="y domain",
                       text=f"median {median_s:.2f}", font=dict(color="steelblue", size=10),
                       showarrow=False, yanchor="top", xanchor="left")
    fig.update_xaxes(range=[0, 1], title_text="IQB Score", row=1, col=2)
    fig.update_yaxes(title_text="# Schools", row=1, col=2)

    # Bar — ranked schools
    bar_colors = [score_color(s, threshold) for s in df_sorted["score"]]
    fig.add_trace(go.Bar(
        x=list(range(len(df_sorted))),
        y=df_sorted["score"],
        marker_color=bar_colors,
        hovertext=df_sorted["school_id"],
        hovertemplate=(
            "<b>School:</b> %{hovertext}<br>"
            "<b>Score:</b> %{y:.3f}<extra></extra>"
        ),
        name="Score",
    ), row=1, col=3)
    fig.add_shape(type="line", x0=0, x1=1, y0=threshold, y1=threshold,
                  xref="x2 domain", yref="y2",
                  line=dict(dash="dash", color="crimson", width=1.5))
    fig.update_xaxes(
        title_text="Schools (low → high score)", showticklabels=False, row=1, col=3,
    )
    fig.update_yaxes(range=[0, 1.05], title_text="IQB Score", row=1, col=3)

    n_ok   = int(round(pct_ok / 100 * len(df)))
    n_fail = len(df) - n_ok
    title  = (
        f"<b>{scenario_dd.value}</b>  ·  {perc_short}  ·  "
        f"<span style='color:#2ca02c'>{n_ok} schools ready</span>  "
        f"<span style='color:#d62728'>/ {n_fail} not ready</span>  "
        f"(threshold {threshold:.2f},  median {median_s:.2f})"
    )
    fig.update_layout(
        height=430,
        showlegend=False,
        margin=dict(t=70, b=40, l=40, r=20),
        title_text=title,
        title_font_size=12,
    )

    # ── Per-use-case breakdown table ─────────────────────────────────────
    rows = []
    for uc in IQB_USE_CASES:
        w = weights[uc]
        if w == 0:
            continue
        uc_scores = compute_uc_scores(uc, percentile)
        med   = float(np.median(uc_scores))
        p_ok  = float(np.mean([s >= 0.5 for s in uc_scores])) * 100
        p_thr = float(np.mean([s >= threshold for s in uc_scores])) * 100
        rows.append({
            "Use Case":              UC_LABELS[uc],
            "Your Priority":         "●" * int(round(w * 5)) + "○" * (5 - int(round(w * 5))),
            "Median School Score":   f"{med:.2f}  ({score_label(med)})",
            "% Schools ≥ 0.50":     f"{p_ok:.0f}%",
            f"% Schools ≥ {threshold:.2f}": f"{p_thr:.0f}%",
        })

    with output:
        clear_output(wait=True)
        fig.show()

        if rows:
            tbl_html = (
                pd.DataFrame(rows)
                .style
                .set_properties(**{"text-align": "center", "padding": "4px 12px"})
                .set_table_styles([
                    {"selector": "th",
                     "props": [("background", "#f0f4f8"), ("text-align", "center"),
                               ("padding", "6px 12px")]},
                ])
                .hide(axis="index")
                .to_html()
            )
            display(HTML(
                "<h4 style='margin-top:18px'>Use Case Breakdown "
                "<span style='font-weight:normal;font-size:0.85em;color:#666'>"
                "(active priorities only)</span></h4>"
                + tbl_html
            ))

        # Plain-language summary
        gap = threshold - median_s
        if gap > 0:
            gap_msg = (
                f"The median school score ({median_s:.2f}) is "
                f"<b>{gap:.2f} points below</b> your threshold. "
                "Improving connectivity across the system would close this gap."
            )
        else:
            gap_msg = (
                f"The median school score ({median_s:.2f}) already "
                f"<b>exceeds</b> your threshold — most schools are on track."
            )
        display(HTML(
            f"<p style='margin-top:12px;color:#333;font-size:0.93em'>"
            f"<b>Reading the results:</b> Under the <em>{scenario_dd.value}</em> scenario, "
            f"<b>{pct_ok:.0f}%</b> of schools ({n_ok} of {len(df)}) meet your minimum score. "
            f"{gap_msg}</p>"
        ))

# ── Wire observers ────────────────────────────────────────────────────────
scenario_dd.observe(load_scenario, names="value")
for sl in weight_sliders.values():
    sl.observe(refresh, names="value")
perc_toggle.observe(refresh, names="value")
threshold_sl.observe(refresh, names="value")

# ── Layout ────────────────────────────────────────────────────────────────
sep = "<hr style='margin:10px 0;border-color:#ddd'>"
controls = widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px'>📋 Policy Scenario</h3>"),
    scenario_dd,
    scenario_desc,

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>⚖️ Use Case Priorities</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "0 = not a priority &nbsp;·&nbsp; 1 = top priority</p>"),
    *[weight_sliders[uc] for uc in IQB_USE_CASES],

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>📅 Performance Standard</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "Which slice of the month's measurements to evaluate</p>"),
    perc_toggle,

    widgets.HTML(sep + "<h3 style='margin:0 0 2px'>📊 IQB-Edu Scores for Schools</h3>"
                 "<p style='margin:0 0 8px;font-size:0.87em;color:#555'>"
                 "Pick a threshold to see how many schools perform above it. "
                 "Schools below the threshold appear red in the ranked chart.</p>"),
    threshold_sl,
])

display(widgets.VBox([controls, output]))
refresh(None)


---
## 🔍 Methodology & Transparency

The IQB-Edu framework converts raw network measurements into educational readiness scores.
The sections below explain each step — expand them to inspect the details.

<details>
<summary><b>How are IQB scores calculated?</b></summary>

### Step 1 — Measurements
M-Lab's NDT test runs from each school's network and records:
- **Download throughput** (Mbps)
- **Upload throughput** (Mbps)
- **Round-trip latency** (ms)
- **Packet loss** (fraction 0–1)

Moldova schools in this dataset were measured throughout January 2026.
Each school has a distribution of results, not a single number.

### Step 2 — Percentiles
Instead of using a single measurement, we pick a **percentile** of the distribution:

| Percentile | What it represents |
|---|---|
| p25 | Conditions during the worst quarter of measurements — a hard day |
| p50 | The median — a typical day |
| p95 | Near-peak — what the connection can do at its best |

Choosing p25 is the most demanding test; p95 the most generous.

### Step 3 — Per-use-case scoring
For each use case (web browsing, video conferencing, etc.) the IQB defines
**minimum network requirements**. A school's score for that use case is 1 if all
requirements are met, 0 if none are, and scales smoothly between those extremes
based on how close each metric comes to its threshold.

### Step 4 — Weighted composite
The final IQB score is a **weighted average** of per-use-case scores.
The weights you set above control how much each use case contributes to the composite.
Setting a weight to 0 removes that use case entirely.

### Caveats
- Schools with fewer than ~30 measurements have noisier scores; treat them as indicative.
- NDT tests reflect actual usage conditions (congestion, ISP throttling), not raw capacity.
- IQB thresholds represent current best-practice standards and can be updated as norms evolve.

</details>

<details>
<summary><b>IQB use-case network requirements (current defaults)</b></summary>


In [5]:
rows = []
for uc, cfg in IQB_CONFIG["use cases"].items():
    req = cfg.get("network requirements", {})
    rows.append({
        "Use Case":        uc.title(),
        "Download (Mbps)": req.get("download_throughput_mbps", {}).get("threshold min", "—"),
        "Upload (Mbps)":   req.get("upload_throughput_mbps",  {}).get("threshold min", "—"),
        "Latency (ms)":    req.get("latency_ms",              {}).get("threshold min", "—"),
        "Packet Loss":     req.get("packet_loss",             {}).get("threshold min", "—"),
        "Default Weight":  cfg.get("w", "—"),
    })
display(pd.DataFrame(rows).style.hide(axis="index"))


Use Case,Download (Mbps),Upload (Mbps),Latency (ms),Packet Loss,Default Weight
Web Browsing,10,10,100,0.010000,1
Video Streaming,25,10,100,0.010000,1
Audio Streaming,10,5,100,0.010000,1
Video Conferencing,25,25,50,0.005000,1
Online Backup,10,25,100,0.010000,1
Gaming,25,25,10,0.005000,1


</details>